# TALLER 3 — FLUJO ANALÍTICO REPRODUCIBLE

### Requisito:
La calificación del Taller Práctico está condicionada a la entrega previa del vídeo semanal.

### Reto
Diseñar un flujo analítico reproducible.

### Objetivo y alcance del reto:
Transformar un proceso manual de consolidación de datos en un flujo reproducible. Se aplica agrupamiento, agregación, combinación de datasets, automatización de archivos, gestión de entornos, organización de proyectos y control de versiones.

### Modalidad del entregable:
Escrito — permite documentar decisiones técnicas, justificar criterios de reproducibilidad, presentar la estructura del proyecto y explicar validaciones de forma ordenada.

### 1.1 Configuración del entorno

In [ ]:
#? Importamos las dependencias mínimas para el flujo completo.
#? sys: verificar versión de Python; pathlib: rutas portables; datetime: sellos de tiempo en exports.
#? logging: sistema formal de registro de eventos; json: carga de configuración externa.
import sys
import json
import logging
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
import shutil
warnings.filterwarnings('ignore')

# --- LOGGING: Sistema formal de registro de eventos ---
#? Reemplazamos prints por logging para control de niveles y persistencia en archivo.
LOG_DIR = Path.cwd() / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(LOG_DIR / 'pipeline.log', mode='w', encoding='utf-8')
    ]
)
logger = logging.getLogger(__name__)

# --- CONFIGURACIÓN EXTERNA: Parámetros centralizados ---
#? Cargamos parámetros desde config.json para evitar valores hardcodeados.
CONFIG_PATH = Path.cwd() / 'config.json'
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    CONFIG = json.load(f)

logger.info('=== INICIO DEL PIPELINE ===')
logger.info(f'Proyecto: {CONFIG["proyecto"]} v{CONFIG["version"]}')
logger.info(f'Python: {sys.version.split()[0]}')
logger.info(f'Pandas: {pd.__version__}')
logger.info(f'Numpy:  {np.__version__}')
logger.info(f'Configuración cargada desde: {CONFIG_PATH.name}')

2026-08-22 22:07:40,401 | INFO     | === INICIO DEL PIPELINE ===
2026-08-22 22:07:40,402 | INFO     | Proyecto: Taller 3 - Flujo Analítico Reproducible v2.0
2026-08-22 22:07:40,402 | INFO     | Python: 3.12.0
2026-08-22 22:07:40,403 | INFO     | Pandas: 3.0.5
2026-08-22 22:07:40,403 | INFO     | Numpy:  2.5.2
2026-08-22 22:07:40,404 | INFO     | Configuración cargada desde: config.json


### 1.2 Arquitectura de carpetas

In [ ]:
#? Rutas definidas en config.json: el proyecto corre en cualquier máquina sin editar paths.
#? Separamos raw (intocable) de processed (generado) para trazabilidad.
BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / CONFIG['rutas']['raw']
PROCESSED_DIR = BASE_DIR / CONFIG['rutas']['processed']

for directorio in [RAW_DIR, PROCESSED_DIR]:
    directorio.mkdir(parents=True, exist_ok=True)

logger.info('--- ESTRUCTURA DE CARPETAS ---')
logger.info(f'Raw:       {RAW_DIR}')
logger.info(f'Processed: {PROCESSED_DIR}')

2026-08-22 22:07:40,411 | INFO     | --- ESTRUCTURA DE CARPETAS ---
2026-08-22 22:07:40,411 | INFO     | Raw:       /Users/josias.pina/G6_Taller_Clase3/data/raw
2026-08-22 22:07:40,412 | INFO     | Processed: /Users/josias.pina/G6_Taller_Clase3/data/processed


### 1.3 Funciones reutilizables

In [ ]:
#? Centralizamos lógica repetitiva en funciones para no duplicar código entre secciones.
#? Cada función encapsula una validación que se aplica a múltiples DataFrames.

def leer_archivo_seguro(ruta, tipo='csv', **kwargs):
    """Lee CSV, Excel o JSON de forma robusta con logging."""
    try:
        if tipo == 'csv':
            df = pd.read_csv(ruta, **kwargs)
        elif tipo == 'excel':
            df = pd.read_excel(ruta, **kwargs)
        elif tipo == 'json':
            df = pd.read_json(ruta, **kwargs)
        else:
            raise ValueError(f'Tipo no soportado: {tipo}')
        logger.info(f'Archivo leído: {Path(ruta).name} ({len(df)} filas, {len(df.columns)} cols)')
        return df
    except Exception as e:
        logger.error(f'Error leyendo {ruta}: {e}')
        raise


def validar_columnas(df, columnas_requeridas, nombre_df='DataFrame'):
    """Falla temprano si faltan columnas esperadas."""
    faltantes = [c for c in columnas_requeridas if c not in df.columns]
    if faltantes:
        logger.error(f'{nombre_df} sin columnas: {faltantes}')
        raise ValueError(f'{nombre_df} sin columnas: {faltantes}')
    logger.info(f'   \u2713 {nombre_df}: columnas validadas')
    return True


def detectar_duplicados(df, subset, nombre_df='DataFrame'):
    """Reporta duplicados sobre un subset de columnas."""
    dupes = df[df.duplicated(subset=subset, keep=False)]
    n = len(dupes)
    if n > 0:
        logger.warning(f'{nombre_df}: {n} registros duplicados en {subset}')
    else:
        logger.info(f'   \u2713 {nombre_df}: 0 duplicados en {subset}')
    return dupes


def contar_nulos(df, columna, nombre_df='DataFrame'):
    """Cuenta nulos en una columna y reporta porcentaje."""
    n = df[columna].isna().sum()
    pct = n / len(df) * 100
    if n > 0:
        logger.warning(f'{nombre_df} \u2192 {columna}: {n} nulos ({pct:.1f}%)')
    else:
        logger.info(f'   \u2713 {nombre_df} \u2192 {columna}: 0 nulos')
    return n

logger.info('--- FUNCIONES CARGADAS ---')
logger.info('leer_archivo_seguro, validar_columnas, detectar_duplicados, contar_nulos')

2026-08-22 22:07:40,420 | INFO     | --- FUNCIONES CARGADAS ---
2026-08-22 22:07:40,422 | INFO     | leer_archivo_seguro, validar_columnas, detectar_duplicados, contar_nulos


### 1.4 Pruebas unitarias automatizadas

In [ ]:
#? Pruebas unitarias: validamos las funciones críticas con datos sintéticos.
#? Si alguna falla, el pipeline se detiene antes de procesar datos reales.
import tempfile
import os

def ejecutar_pruebas():
    """Suite de pruebas unitarias para funciones del pipeline."""
    errores = []
    pruebas_pasadas = 0

    # --- Test 1: leer_archivo_seguro con CSV válido ---
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as f:
            f.write('col_a,col_b\n1,2\n3,4\n')
            tmp_path = f.name
        df_test = leer_archivo_seguro(tmp_path, tipo='csv')
        assert len(df_test) == 2, f'Esperado 2 filas, obtenido {len(df_test)}'
        assert list(df_test.columns) == ['col_a', 'col_b'], 'Columnas incorrectas'
        os.unlink(tmp_path)
        pruebas_pasadas += 1
    except AssertionError as e:
        errores.append(f'test_leer_csv_valido: {e}')

    # --- Test 2: leer_archivo_seguro con tipo inválido ---
    try:
        error_capturado = False
        try:
            leer_archivo_seguro('dummy.txt', tipo='xml')
        except ValueError:
            error_capturado = True
        assert error_capturado, 'Debería lanzar ValueError para tipo no soportado'
        pruebas_pasadas += 1
    except AssertionError as e:
        errores.append(f'test_leer_tipo_invalido: {e}')

    # --- Test 3: validar_columnas con columnas presentes ---
    try:
        df_test = pd.DataFrame({'a': [1], 'b': [2], 'c': [3]})
        resultado = validar_columnas(df_test, ['a', 'b'], 'Test')
        assert resultado == True, 'Debería retornar True'
        pruebas_pasadas += 1
    except AssertionError as e:
        errores.append(f'test_validar_columnas_ok: {e}')

    # --- Test 4: validar_columnas con columnas faltantes ---
    try:
        df_test = pd.DataFrame({'a': [1], 'b': [2]})
        error_capturado = False
        try:
            validar_columnas(df_test, ['a', 'x', 'y'], 'Test')
        except ValueError:
            error_capturado = True
        assert error_capturado, 'Debería lanzar ValueError'
        pruebas_pasadas += 1
    except AssertionError as e:
        errores.append(f'test_validar_columnas_faltantes: {e}')

    # --- Test 5: detectar_duplicados ---
    try:
        df_test = pd.DataFrame({'id': [1, 2, 2, 3], 'val': ['a', 'b', 'c', 'd']})
        dupes = detectar_duplicados(df_test, subset=['id'], nombre_df='Test')
        assert len(dupes) == 2, f'Esperado 2 duplicados, obtenido {len(dupes)}'
        pruebas_pasadas += 1
    except AssertionError as e:
        errores.append(f'test_detectar_duplicados: {e}')

    # --- Test 6: detectar_duplicados sin duplicados ---
    try:
        df_test = pd.DataFrame({'id': [1, 2, 3], 'val': ['a', 'b', 'c']})
        dupes = detectar_duplicados(df_test, subset=['id'], nombre_df='Test')
        assert len(dupes) == 0, f'Esperado 0 duplicados, obtenido {len(dupes)}'
        pruebas_pasadas += 1
    except AssertionError as e:
        errores.append(f'test_sin_duplicados: {e}')

    # --- Test 7: contar_nulos con nulos ---
    try:
        df_test = pd.DataFrame({'score': [10, None, 30, None, 50]})
        n = contar_nulos(df_test, 'score', 'Test')
        assert n == 2, f'Esperado 2 nulos, obtenido {n}'
        pruebas_pasadas += 1
    except AssertionError as e:
        errores.append(f'test_contar_nulos: {e}')

    # --- Test 8: contar_nulos sin nulos ---
    try:
        df_test = pd.DataFrame({'score': [10, 20, 30]})
        n = contar_nulos(df_test, 'score', 'Test')
        assert n == 0, f'Esperado 0 nulos, obtenido {n}'
        pruebas_pasadas += 1
    except AssertionError as e:
        errores.append(f'test_sin_nulos: {e}')

    return pruebas_pasadas, errores

# Ejecutar pruebas
logger.info('=== PRUEBAS UNITARIAS ===')
total_pasadas, errores_encontrados = ejecutar_pruebas()
total_tests = total_pasadas + len(errores_encontrados)

if errores_encontrados:
    for err in errores_encontrados:
        logger.error(f'FALLO: {err}')
    raise RuntimeError(f'{len(errores_encontrados)}/{total_tests} pruebas fallaron. Pipeline detenido.')
else:
    logger.info(f'\u2713 {total_pasadas}/{total_tests} pruebas pasaron exitosamente')
    logger.info('Pipeline validado para continuar.')

2026-08-22 22:07:40,436 | INFO     | === PRUEBAS UNITARIAS ===
2026-08-22 22:07:40,462 | INFO     | Archivo leído: tmpup3yfejj.csv (2 filas, 2 cols)
2026-08-22 22:07:40,464 | ERROR    | Error leyendo dummy.txt: Tipo no soportado: xml
2026-08-22 22:07:40,465 | INFO     |    ✓ Test: columnas validadas
2026-08-22 22:07:40,466 | ERROR    | Test sin columnas: ['x', 'y']
2026-08-22 22:07:40,473 | WARNING  | Test: 2 registros duplicados en ['id']
2026-08-22 22:07:40,475 | INFO     |    ✓ Test: 0 duplicados en ['id']
2026-08-22 22:07:40,476 | WARNING  | Test → score: 2 nulos (40.0%)
2026-08-22 22:07:40,477 | INFO     |    ✓ Test → score: 0 nulos
2026-08-22 22:07:40,478 | INFO     | ✓ 8/8 pruebas pasaron exitosamente
2026-08-22 22:07:40,478 | INFO     | Pipeline validado para continuar.


In [ ]:
#? Copiamos los archivos fuente al directorio raw para garantizar trazabilidad.
#? Los nombres de archivos se obtienen de config.json.

logger.info('--- COPIADO DE ARCHIVOS A RAW ---')
archivos_fuente = CONFIG['archivos_fuente']

for clave, nombre in archivos_fuente.items():
    origen = BASE_DIR / nombre
    destino = RAW_DIR / nombre
    if origen.exists():
        shutil.copy2(origen, destino)
        logger.info(f'  \u2713 {nombre} copiado a {CONFIG["rutas"]["raw"]}/')
    else:
        logger.warning(f'  \u2717 {nombre} NO encontrado en {origen}')

logger.info('--- FIN COPIADO ---')

2026-08-22 22:07:40,485 | INFO     | --- COPIADO DE ARCHIVOS A RAW ---
2026-08-22 22:07:40,492 | INFO     |   ✓ estudiantes_master.xlsx copiado a data/raw/
2026-08-22 22:07:40,500 | INFO     |   ✓ entregas_campus_matriz.csv copiado a data/raw/
2026-08-22 22:07:40,508 | INFO     |   ✓ entregas_campus_extension.csv copiado a data/raw/
2026-08-22 22:07:40,509 | INFO     | --- FIN COPIADO ---


### 2.1 Lectura estructurada de datos

In [ ]:
#? Forzamos dtype str en IDs para evitar que pandas los interprete como numéricos y pierda ceros.
#? Columnas requeridas se obtienen de config.json.

logger.info('--- CARGA DE DATOS ---')

# Catálogo maestro (Excel)
ruta_catalogo = RAW_DIR / CONFIG['archivos_fuente']['catalogo']
try:
    df_catalogo = leer_archivo_seguro(
        ruta_catalogo, tipo='excel', engine='openpyxl',
        dtype={'id_estudiante': str}
    )
except Exception:
    df_catalogo = leer_archivo_seguro(
        ruta_catalogo, tipo='csv', sep=',',
        dtype={'id_estudiante': str}
    )

columnas_catalogo = CONFIG['columnas_requeridas']['catalogo']
validar_columnas(df_catalogo, columnas_catalogo, 'Cat\u00e1logo')

#? Eliminamos duplicados en la llave primaria antes del merge.
df_catalogo_limpio = df_catalogo.drop_duplicates(subset=['id_estudiante'])
logger.info(f'   Cat\u00e1logo: {len(df_catalogo)} \u2192 {len(df_catalogo_limpio)} (sin duplicados)')

# Entregas campus Matriz (CSV)
df_matriz = leer_archivo_seguro(
    RAW_DIR / CONFIG['archivos_fuente']['entregas_matriz'],
    tipo='csv', sep=',',
    dtype={'id_entrega': str, 'id_estudiante': str}
)

# Entregas campus Extensión (CSV)
df_extension = leer_archivo_seguro(
    RAW_DIR / CONFIG['archivos_fuente']['entregas_extension'],
    tipo='csv', sep=',',
    dtype={'id_entrega': str, 'id_estudiante': str}
)

columnas_entregas = CONFIG['columnas_requeridas']['entregas']
validar_columnas(df_matriz, columnas_entregas, 'Matriz')
validar_columnas(df_extension, columnas_entregas, 'Extensi\u00f3n')

logger.info(f'   Matriz:    {df_matriz.shape}')
logger.info(f'   Extensi\u00f3n: {df_extension.shape}')
logger.info(f'   Cat\u00e1logo:  {df_catalogo_limpio.shape}')

2026-08-22 22:07:40,516 | INFO     | --- CARGA DE DATOS ---
2026-08-22 22:07:40,525 | ERROR    | Error leyendo /Users/josias.pina/G6_Taller_Clase3/data/raw/estudiantes_master.xlsx: File is not a zip file
2026-08-22 22:07:40,529 | INFO     | Archivo leído: estudiantes_master.xlsx (60 filas, 7 cols)
2026-08-22 22:07:40,530 | INFO     |    ✓ Catálogo: columnas validadas
2026-08-22 22:07:40,532 | INFO     |    Catálogo: 60 → 60 (sin duplicados)
2026-08-22 22:07:40,535 | INFO     | Archivo leído: entregas_campus_matriz.csv (150 filas, 7 cols)
2026-08-22 22:07:40,538 | INFO     | Archivo leído: entregas_campus_extension.csv (150 filas, 7 cols)
2026-08-22 22:07:40,538 | INFO     |    ✓ Matriz: columnas validadas
2026-08-22 22:07:40,539 | INFO     |    ✓ Extensión: columnas validadas
2026-08-22 22:07:40,539 | INFO     |    Matriz:    (150, 7)
2026-08-22 22:07:40,539 | INFO     |    Extensión: (150, 7)
2026-08-22 22:07:40,540 | INFO     |    Catálogo:  (60, 7)


### 2.2 Integración estructural (concat)

In [ ]:
#? Concat: unimos DataFrames con esquema idéntico (mismas columnas, distinto origen).
#? Agregamos columna 'campus' ANTES de concatenar para no perder el origen de cada fila.

df_matriz['campus'] = 'matriz'
df_extension['campus'] = 'extension'

df_consolidado = pd.concat(
    [df_matriz, df_extension],
    ignore_index=True
)

#? Verificamos que no haya IDs de entrega repetidos entre campus.
detectar_duplicados(df_consolidado, subset=['id_entrega'], nombre_df='Consolidado')

logger.info(f'--- CONCAT ---')
logger.info(f'Matriz ({len(df_matriz)}) + Extensi\u00f3n ({len(df_extension)}) = {len(df_consolidado)} registros')

2026-08-22 22:07:40,548 | INFO     |    ✓ Consolidado: 0 duplicados en ['id_entrega']
2026-08-22 22:07:40,549 | INFO     | --- CONCAT ---
2026-08-22 22:07:40,549 | INFO     | Matriz (150) + Extensión (150) = 300 registros


### 2.3 Integración relacional (merge)

In [ ]:
#? Merge left: conservamos TODAS las entregas aunque el estudiante no esté en el catálogo.
#? validate='many_to_one': falla si el catálogo tiene claves duplicadas (detecta error upstream).
#? indicator=True: nos permite auditar qué registros no encontraron match.

df_integrado = pd.merge(
    df_consolidado,
    df_catalogo_limpio,
    on='id_estudiante',
    how='left',
    validate='many_to_one',
    indicator=True
)

logger.info('--- MERGE ---')
logger.info(df_integrado['_merge'].value_counts().to_string())

#? Huérfanos = entregas de estudiantes no matriculados. Los exportamos para auditoría.
df_huerfanos = df_integrado[df_integrado['_merge'] == 'left_only']
logger.info(f'Hu\u00e9rfanos: {len(df_huerfanos)} ({len(df_huerfanos)/len(df_integrado)*100:.1f}%)')

archivos_exportados = []  # Inicializamos el contador global de archivos

if len(df_huerfanos) > 0:
    archivo_huerfanos = PROCESSED_DIR / f'huerfanos_{datetime.now().strftime("%Y%m%d")}.csv'
    df_huerfanos.to_csv(archivo_huerfanos, index=False)
    archivos_exportados.append(archivo_huerfanos.name)
    logger.info(f'   Exportado \u2192 {archivo_huerfanos.name}')

df_integrado = df_integrado.drop(columns=['_merge'])

2026-08-22 22:07:40,569 | INFO     | --- MERGE ---
2026-08-22 22:07:40,571 | INFO     | _merge
both          295
left_only       5
right_only      0
2026-08-22 22:07:40,573 | INFO     | Huérfanos: 5 (1.7%)
2026-08-22 22:07:40,577 | INFO     |    Exportado → huerfanos_20260822.csv


### 2.4 Validaciones adicionales y limpieza

In [ ]:
#? Identificamos nulos críticos en puntaje: pueden indicar entregas no calificadas.
#? También buscamos inconsistencias lógicas entre estado y puntaje.

logger.info('--- VALIDACIONES ---')
nulos_puntaje = contar_nulos(df_integrado, 'puntaje_obtenido', 'Integrado')

# Inconsistencias: Pendiente con puntaje o Revisión sin puntaje
pendiente_con_nota = df_integrado[
    (df_integrado['estado_entrega'] == 'Pendiente') &
    (df_integrado['puntaje_obtenido'].notna())
]
revision_sin_nota = df_integrado[
    (df_integrado['estado_entrega'] == 'Revisi\u00f3n') &
    (df_integrado['puntaje_obtenido'].isna())
]

logger.info(f'   Pendiente CON puntaje (inconsistente): {len(pendiente_con_nota)}')
logger.info(f'   Revisi\u00f3n SIN puntaje (inconsistente):  {len(revision_sin_nota)}')

2026-08-22 22:07:40,584 | INFO     | --- VALIDACIONES ---
2026-08-22 22:07:40,585 | WARNING  | Integrado → puntaje_obtenido: 9 nulos (3.0%)
2026-08-22 22:07:40,587 | INFO     |    Pendiente CON puntaje (inconsistente): 0
2026-08-22 22:07:40,587 | INFO     |    Revisión SIN puntaje (inconsistente):  0


### 2.5 Análisis exploratorio y agregaciones

In [ ]:
#? Agregamos por múltiples ejes para entender la distribución de los datos.

logger.info('--- DESCRIPTIVOS ---')
logger.info('\n' + df_integrado['puntaje_obtenido'].describe().to_string())

logger.info('--- PROMEDIO POR CAMPUS ---')
logger.info('\n' + df_integrado.groupby('campus')['puntaje_obtenido'].mean().to_string())

logger.info('--- ESTADO DE ENTREGAS ---')
logger.info('\n' + df_integrado['estado_entrega'].value_counts().to_string())

logger.info('--- TOP 5 ESTUDIANTES (promedio) ---')
top5 = (df_integrado.groupby('id_estudiante')['puntaje_obtenido']
        .mean().nlargest(5))
logger.info('\n' + top5.to_string())

logger.info('--- BOTTOM 5 ESTUDIANTES (promedio) ---')
bottom5 = (df_integrado.groupby('id_estudiante')['puntaje_obtenido']
           .mean().nsmallest(5))
logger.info('\n' + bottom5.to_string())

logger.info('--- PROMEDIO POR MATERIA ---')
logger.info('\n' + df_integrado.groupby('materia')['puntaje_obtenido'].mean().sort_values(ascending=False).to_string())

logger.info('--- PROMEDIO POR TIPO DE PROYECTO ---')
logger.info('\n' + df_integrado.groupby('tipo_proyecto')['puntaje_obtenido'].mean().sort_values(ascending=False).to_string())

2026-08-22 22:07:40,595 | INFO     | --- DESCRIPTIVOS ---
2026-08-22 22:07:40,601 | INFO     | 
count    291.000000
mean      90.347079
std        6.627905
min       60.000000
25%       87.000000
50%       91.000000
75%       95.000000
max      100.000000
2026-08-22 22:07:40,603 | INFO     | --- PROMEDIO POR CAMPUS ---
2026-08-22 22:07:40,606 | INFO     | 
campus
extension    90.697279
matriz       89.989583
2026-08-22 22:07:40,607 | INFO     | --- ESTADO DE ENTREGAS ---
2026-08-22 22:07:40,609 | INFO     | 
estado_entrega
Aprobado     276
Revisión      15
Pendiente      9
2026-08-22 22:07:40,610 | INFO     | --- TOP 5 ESTUDIANTES (promedio) ---
2026-08-22 22:07:40,612 | INFO     | 
id_estudiante
E999    100.000000
E026     99.285714
E058     99.000000
E039     98.500000
E043     98.333333
2026-08-22 22:07:40,613 | INFO     | --- BOTTOM 5 ESTUDIANTES (promedio) ---
2026-08-22 22:07:40,615 | INFO     | 
id_estudiante
E888    60.000000
E012    70.333333
E030    76.714286
E020    81.80000

### 2.6 Resumen analítico por estudiante

In [ ]:
#? Agrupamos por estudiante+campus para generar un perfil resumido por persona.
#? La desviación estándar revela consistencia: un alumno con std alta es irregular.

resumen_final = (df_integrado
    .groupby(['id_estudiante', 'campus'])
    .agg(
        promedio=('puntaje_obtenido', 'mean'),
        entregas=('id_entrega', 'count'),
        desviacion=('puntaje_obtenido', 'std')
    )
    .reset_index()
)

logger.info('--- RESUMEN POR ESTUDIANTE ---')
logger.info(f'Dimensiones: {resumen_final.shape}')
logger.info('\n' + resumen_final.head(10).to_string(index=False))

2026-08-22 22:07:40,631 | INFO     | --- RESUMEN POR ESTUDIANTE ---
2026-08-22 22:07:40,632 | INFO     | Dimensiones: (64, 5)
2026-08-22 22:07:40,635 | INFO     | 
id_estudiante    campus  promedio  entregas  desviacion
         E001    matriz 93.785714         7    4.376615
         E002    matriz 83.857143         7    7.358183
         E003    matriz 91.600000         7    1.557241
         E004 extension 89.857143         8    1.772811
         E005 extension 94.250000         8    1.982062
         E006    matriz 96.857143         7    3.023716
         E007    matriz 85.500000         7    4.330127
         E008    matriz 94.142857         7    3.023716
         E009 extension 87.562500         8    1.678381
         E010 extension 97.875000         8    1.246423


### 2.7 Indicadores de calidad del proceso

In [ ]:
#? Los indicadores auditan el pipeline: si alguno cambia entre ejecuciones, algo se rompió.
#? MEJORA: Métricas de completitud, consistencia y unicidad detalladas.

logger.info('=== INDICADORES DE CALIDAD DEL PROCESO ===')

# --- Métricas básicas del pipeline ---
indicadores_basicos = {
    'archivos_procesados': len(CONFIG['archivos_fuente']),
    'registros_catalogo': len(df_catalogo),
    'registros_matriz': len(df_matriz),
    'registros_extension': len(df_extension),
    'registros_consolidados': len(df_consolidado),
    'registros_integrados': len(df_integrado),
    'duplicados_catalogo_eliminados': len(df_catalogo) - len(df_catalogo_limpio),
    'claves_sin_correspondencia': len(df_huerfanos),
    'nulos_puntaje_obtenido': nulos_puntaje,
    'dimension_salida_filas': df_integrado.shape[0],
    'dimension_salida_columnas': df_integrado.shape[1],
}

# --- COMPLETITUD: % de celdas no-nulas por columna ---
logger.info('\n--- M\u00c9TRICAS DE COMPLETITUD ---')
completitud = {}
for col in df_integrado.columns:
    no_nulos = df_integrado[col].notna().sum()
    total = len(df_integrado)
    pct = (no_nulos / total) * 100
    completitud[f'completitud_{col}'] = round(pct, 2)
    logger.info(f'   {col:.<35} {pct:.2f}%')

completitud_global = sum(completitud.values()) / len(completitud)
logger.info(f'   {"COMPLETITUD GLOBAL":.<35} {completitud_global:.2f}%')

# --- CONSISTENCIA: Reglas de negocio ---
logger.info('\n--- M\u00c9TRICAS DE CONSISTENCIA ---')
puntaje_min = CONFIG['validaciones']['puntaje_minimo']
puntaje_max = CONFIG['validaciones']['puntaje_maximo']
estados_validos = CONFIG['validaciones']['estados_validos']

# Puntajes fuera de rango
fuera_rango = df_integrado[
    (df_integrado['puntaje_obtenido'].notna()) &
    ((df_integrado['puntaje_obtenido'] < puntaje_min) |
     (df_integrado['puntaje_obtenido'] > puntaje_max))
]
# Estados inválidos
estados_invalidos = df_integrado[
    ~df_integrado['estado_entrega'].isin(estados_validos)
]

consistencia = {
    'puntajes_fuera_rango': len(fuera_rango),
    'estados_invalidos': len(estados_invalidos),
    'pendiente_con_puntaje': len(pendiente_con_nota),
    'revision_sin_puntaje': len(revision_sin_nota),
    'consistencia_puntaje_pct': round((1 - len(fuera_rango) / len(df_integrado)) * 100, 2),
    'consistencia_estados_pct': round((1 - len(estados_invalidos) / len(df_integrado)) * 100, 2),
}

for k, v in consistencia.items():
    logger.info(f'   {k:.<35} {v}')

# --- UNICIDAD: Valores \u00fanicos en columnas clave ---
logger.info('\n--- M\u00c9TRICAS DE UNICIDAD ---')
unicidad = {}
for col in ['id_entrega', 'id_estudiante']:
    total = len(df_integrado)
    unicos = df_integrado[col].nunique()
    duplicados = total - unicos
    pct_unicidad = (unicos / total) * 100
    unicidad[f'unicidad_{col}_unicos'] = unicos
    unicidad[f'unicidad_{col}_duplicados'] = duplicados
    unicidad[f'unicidad_{col}_pct'] = round(pct_unicidad, 2)
    logger.info(f'   {col}: {unicos} \u00fanicos / {total} total ({pct_unicidad:.2f}% unicidad)')

# --- Consolidar todos los indicadores ---
indicadores = {
    **indicadores_basicos,
    'completitud_global_pct': round(completitud_global, 2),
    **consistencia,
    **unicidad,
}

logger.info('\n--- RESUMEN DE INDICADORES ---')
for k, v in indicadores.items():
    logger.info(f'   {k:.<40} {v}')

2026-08-22 22:07:40,648 | INFO     | === INDICADORES DE CALIDAD DEL PROCESO ===
2026-08-22 22:07:40,649 | INFO     | 
--- MÉTRICAS DE COMPLETITUD ---
2026-08-22 22:07:40,650 | INFO     |    id_entrega......................... 100.00%
2026-08-22 22:07:40,652 | INFO     |    id_estudiante...................... 100.00%
2026-08-22 22:07:40,653 | INFO     |    fecha_subida....................... 100.00%
2026-08-22 22:07:40,653 | INFO     |    materia............................ 100.00%
2026-08-22 22:07:40,654 | INFO     |    tipo_proyecto...................... 100.00%
2026-08-22 22:07:40,655 | INFO     |    puntaje_obtenido................... 97.00%
2026-08-22 22:07:40,655 | INFO     |    estado_entrega..................... 100.00%
2026-08-22 22:07:40,657 | INFO     |    campus............................. 100.00%
2026-08-22 22:07:40,657 | INFO     |    nombre_completo.................... 98.33%
2026-08-22 22:07:40,658 | INFO     |    fecha_nacimiento................... 98.33%
2026-08-22 22

### 2.8 Exportación de datos finales

In [ ]:
#? Exportamos en 3 formatos definidos en config.json para cubrir distintos consumidores.
#? CSV: universal. JSON: APIs y NoSQL. Excel: usuarios no técnicos.

stamp = datetime.now().strftime('%Y%m%d')
formatos = CONFIG['formatos_exportacion']

logger.info('--- EXPORTACI\u00d3N ---')

# Dataset integrado
for fmt in formatos:
    nombre = PROCESSED_DIR / f'dataset_integrado_{stamp}.{fmt}'
    if fmt == 'json':
        df_integrado.to_json(nombre, orient='records', force_ascii=False, indent=2)
    elif fmt == 'xlsx':
        df_integrado.to_excel(nombre, index=False, engine='openpyxl')
    else:
        df_integrado.to_csv(nombre, index=False)
    archivos_exportados.append(nombre.name)
    logger.info(f'   \u2713 {nombre.name}')

# Resumen por estudiante
for fmt in formatos:
    nombre = PROCESSED_DIR / f'resumen_estudiantes_{stamp}.{fmt}'
    if fmt == 'json':
        resumen_final.to_json(nombre, orient='records', force_ascii=False, indent=2)
    elif fmt == 'xlsx':
        resumen_final.to_excel(nombre, index=False, engine='openpyxl')
    else:
        resumen_final.to_csv(nombre, index=False)
    archivos_exportados.append(nombre.name)
    logger.info(f'   \u2713 {nombre.name}')

# Exportar indicadores de calidad
nombre_ind = PROCESSED_DIR / f'indicadores_calidad_{stamp}.csv'
df_indicadores = pd.DataFrame([indicadores])
df_indicadores.to_csv(nombre_ind, index=False)
archivos_exportados.append(nombre_ind.name)
logger.info(f'   \u2713 {nombre_ind.name}')

# --- CORRECCI\u00d3N: archivos_generados se calcula din\u00e1micamente DESPU\u00c9S de la exportaci\u00f3n ---
#? Contamos archivos reales en processed/ para garantía de auditoría consistente.
archivos_reales = list(PROCESSED_DIR.glob('*'))
indicadores['archivos_generados'] = len(archivos_reales)

# Re-exportar indicadores con el valor correcto
df_indicadores = pd.DataFrame([indicadores])
df_indicadores.to_csv(nombre_ind, index=False)

logger.info(f'\n   Total archivos generados: {len(archivos_reales)} (verificado en disco)')
logger.info(f'   Archivos rastreados en pipeline: {len(archivos_exportados)}')
logger.info('=== EXPORTACI\u00d3N COMPLETA ===')

2026-08-22 22:07:40,685 | INFO     | --- EXPORTACIÓN ---
2026-08-22 22:07:40,690 | INFO     |    ✓ dataset_integrado_20260822.csv
2026-08-22 22:07:40,693 | INFO     |    ✓ dataset_integrado_20260822.json
2026-08-22 22:07:40,762 | INFO     |    ✓ dataset_integrado_20260822.xlsx
2026-08-22 22:07:40,764 | INFO     |    ✓ resumen_estudiantes_20260822.csv
2026-08-22 22:07:40,766 | INFO     |    ✓ resumen_estudiantes_20260822.json
2026-08-22 22:07:40,790 | INFO     |    ✓ resumen_estudiantes_20260822.xlsx
2026-08-22 22:07:40,799 | INFO     |    ✓ indicadores_calidad_20260822.csv
2026-08-22 22:07:40,811 | INFO     | 
   Total archivos generados: 8 (verificado en disco)
2026-08-22 22:07:40,811 | INFO     |    Archivos rastreados en pipeline: 8
2026-08-22 22:07:40,812 | INFO     | === EXPORTACIÓN COMPLETA ===


### 2.9 Estrategia de reproducibilidad

In [ ]:
#? Documentamos las condiciones para que otro analista reproduzca el resultado sin ayuda verbal.

logger.info("""=== ESTRATEGIA DE REPRODUCIBILIDAD ===

1. Entorno virtual:
   python -m venv .venv
   source .venv/bin/activate

2. Dependencias (requirements.txt):
   pandas>=2.2.2
   numpy>=2.2.2
   openpyxl>=3.1.5

3. Configuraci\u00f3n externa:
   - config.json: Par\u00e1metros centralizados del pipeline
   - Modificar config.json para adaptar a otro entorno

4. Control de versiones (Git):
   - .gitignore: data/raw/*, data/processed/*, *.pyc, .venv/, .ipynb_checkpoints/, logs/
   - Commits frecuentes con mensajes descriptivos

5. Ejecuci\u00f3n:
   - Rutas relativas (pathlib) \u2192 portable
   - Para Colab: montar Drive y copiar archivos a data/raw/

6. Validaci\u00f3n:
   - Pruebas unitarias se ejecutan al inicio del pipeline
   - Logging completo en logs/pipeline.log
   - Indicadores de calidad exportados autom\u00e1ticamente

7. Determinismo:
   - Sin operaciones aleatorias
   - Timestamps en nombres de archivos para trazabilidad
""")

logger.info('=== PIPELINE COMPLETADO EXITOSAMENTE ===')

2026-08-22 22:07:40,817 | INFO     | === ESTRATEGIA DE REPRODUCIBILIDAD ===

1. Entorno virtual:
   python -m venv .venv
   source .venv/bin/activate

2. Dependencias (requirements.txt):
   pandas>=2.2.2
   numpy>=2.2.2
   openpyxl>=3.1.5

3. Configuración externa:
   - config.json: Parámetros centralizados del pipeline
   - Modificar config.json para adaptar a otro entorno

4. Control de versiones (Git):
   - .gitignore: data/raw/*, data/processed/*, *.pyc, .venv/, .ipynb_checkpoints/, logs/
   - Commits frecuentes con mensajes descriptivos

5. Ejecución:
   - Rutas relativas (pathlib) → portable
   - Para Colab: montar Drive y copiar archivos a data/raw/

6. Validación:
   - Pruebas unitarias se ejecutan al inicio del pipeline
   - Logging completo en logs/pipeline.log
   - Indicadores de calidad exportados automáticamente

7. Determinismo:
   - Sin operaciones aleatorias
   - Timestamps en nombres de archivos para trazabilidad

2026-08-22 22:07:40,818 | INFO     | === PIPELINE COMPL

### 3.0 Reflexión final

**¿Cómo contribuye el flujo propuesto a la claridad del análisis?**

La separación en secciones (carga → validación → integración → análisis → exportación) permite leer el notebook como un documento lineal donde cada paso tiene un propósito claro. Las funciones reutilizables eliminan código repetido y hacen explícitas las reglas de validación que de otro modo quedarían implícitas en operaciones sueltas.

**¿Qué decisiones favorecen la eficiencia del proceso?**

Usar `validate='many_to_one'` en el merge detecta errores en el catálogo antes de que contaminen el dataset. Exportar en múltiples formatos con un loop evita código duplicado. Las rutas con pathlib y los timestamps en archivos eliminan la necesidad de renombrar manualmente.

**¿Qué elementos garantizan la trazabilidad?**

La columna 'campus' añadida antes del concat preserva el origen de cada fila. El indicador `_merge` identifica huérfanos. Los indicadores de calidad funcionan como un log de auditoría: si en la próxima corrida el número de huérfanos cambia de 5 a 20, sabemos que el catálogo se desactualizó.

**¿Qué riesgos aparecerían si este proceso se hiciera manualmente en una hoja de cálculo?**

Copiar y pegar entre hojas no deja registro de qué filas se unieron ni cuáles quedaron fuera. Un VLOOKUP roto falla silenciosamente devolviendo #N/A sin cuantificar el impacto. No hay versionamiento: si alguien sobrescribe el archivo, se pierde el estado anterior. Escalar a más campus o semestres implicaría repetir manualmente cada paso, multiplicando la probabilidad de error humano.